In [ ]:
!pip install git+https://github.com/NREL/OCHRE.git

!pip install "numpy<2.0"

!pip uninstall gurobipy -y
!pip install gurobipy==12.0.0

print("\nRestart Session")

In [ ]:
import os
import shutil
from ochre.utils import default_input_path
########################################################################################################

source_xml = os.path.join(default_input_path, 'Input Files', 'bldg0112631-up11.xml')
destination_xml = '/content/Default.xml'

shutil.copy(source_xml, destination_xml)
print("Δημιουργήθηκε Default.xml που είναι το xml του ochre.")


In [ ]:
import xml.etree.ElementTree as ET
########################################################################################################

HPXML_NS = "http://hpxmlonline.com/2023/09"
ET.register_namespace("", HPXML_NS)
NS = {"hpxml": HPXML_NS}

LOCATION_CONFIGURATION = {"ATH": {"zipcode": "10431", "epw_file": "GRC_Athens.167160_IWEC.epw"},
                          "THESS": {"zipcode": "54622", "epw_file": "GRC_Thessaloniki.166220_IWEC.epw"}}

BUILDING_SPECS = {
    "HOUSE1": {"U": 93, "C": 27 * 1e6},
    "HOUSE2": {"U": 150, "C": 40 * 1e6},
    "HOUSE3": {"U": 80, "C": 17 * 1e6}}

BUILDING_LAYOUT = {
    "HOUSE1": {"floors_ath": 2, "floors_thess": 2, "bedrooms": 2},
    "HOUSE2": {"floors_ath": 2, "floors_thess": 2, "bedrooms": 2},
    "HOUSE3": {"floors_ath": 2, "floors_thess": 2, "bedrooms": 2}}

HOUSE1_C_PER_U = BUILDING_SPECS["HOUSE1"]["C"] / BUILDING_SPECS["HOUSE1"]["U"]

########################################################################################################

def thermal_mass_from_c(spec):
    capacity_per_area = spec["C"] / spec["U"]
    return 4.0 * (capacity_per_area / HOUSE1_C_PER_U)

########################################################################################################

def generate_house_xml(template_xml, output_xml, area_m2, floors, bedrooms, thermal_mass, location):
    tree = ET.parse(template_xml)
    root = tree.getroot()
    config = LOCATION_CONFIGURATION[location]

    # Location
    state = root.find(".//hpxml:StateCode", NS)
    zipcode = root.find(".//hpxml:ZipCode", NS)
    epw_path = root.find(".//hpxml:extension/hpxml:EPWFilePath", NS)

    state.text = "GR"
    zipcode.text = config["zipcode"]
    epw_path.text = config["epw_file"]

    # Scale area
    floor_area = root.find(".//hpxml:ConditionedFloorArea", NS)
    target_area_sqft = area_m2 * 10.7639
    scale_factor = target_area_sqft / float(floor_area.text)
    floor_area.text = str(round(target_area_sqft, 2))

    conditioned_building_volume = root.find(".//hpxml:ConditionedBuildingVolume", NS)
    conditioned_building_volume.text = str(round(float(conditioned_building_volume.text) * scale_factor, 2))

    for surface_area in root.findall(".//hpxml:Area", NS):
        surface_area.text = str(round(float(surface_area.text) * scale_factor, 2))

    # Building
    stories = root.find(".//hpxml:NumberofConditionedFloorsAboveGrade", NS)
    bedrooms_xml = root.find(".//hpxml:NumberofBedrooms", NS)

    stories.text = str(floors)
    bedrooms_xml.text = str(bedrooms)

    # Thermal mass
    temperature_capacitance_multiplier = root.find(".//hpxml:TemperatureCapacitanceMultiplier", NS)
    temperature_capacitance_multiplier.text = str(thermal_mass)

    # Windows
    for window_ufactor in root.findall(".//hpxml:Window/hpxml:UFactor", NS):
        window_ufactor.text = "0.28"

    tree.write(output_xml, encoding="UTF-8", xml_declaration=True)
    print("Created:", output_xml)

########################################################################################################

# Athens
print("Athens houses...")
for house, spec in BUILDING_SPECS.items():
    layout = BUILDING_LAYOUT[house]
    generate_house_xml('/content/Default.xml', f"/content/{house.lower()}_ath.xml", area_m2=spec["U"], floors=layout["floors_ath"], bedrooms=layout["bedrooms"], thermal_mass=thermal_mass_from_c(spec), location='ATH')

# Thessaloniki
print("Thessaloniki houses...")
for house, spec in BUILDING_SPECS.items():
    layout = BUILDING_LAYOUT[house]
    generate_house_xml('/content/Default.xml', f"/content/{house.lower()}_thess.xml", area_m2=spec["U"], floors=layout["floors_thess"], bedrooms=layout["bedrooms"], thermal_mass=thermal_mass_from_c(spec), location='THESS')

In [ ]:
import os
import numpy as np
import pandas as pd
import datetime as dt
from ochre import Dwelling
from pvlib.iotools import read_epw
from ochre.utils import default_input_path
########################################################################################################

dwelling = Dwelling(start_time=dt.datetime(2018, 1, 1, 0, 0), time_res=dt.timedelta(minutes=60), duration=dt.timedelta(days=365), hpxml_file='/content/house1_thess.xml', weather_file='/content/GRC_Thessaloniki.166220_IWEC.epw')

df, metrics, hourly = dwelling.simulate()

########################################################################################################

# EXPORTS

output_dir = 'HOUSE1'
os.makedirs(output_dir, exist_ok=True)

# 1. Ηλεκτρισμός
pd.DataFrame({
    'datetime': df.index,
    'load_W': df['Total Electric Power (kW)'].values * 1000
}).to_csv(os.path.join(output_dir, 'ELECTRICITY_THESS.csv'), index=False)

########################################################################################################

# 2. Ζεστό Νερό
pd.DataFrame({
    'datetime': df.index,
    'dhw_W': df['Water Heating Electric Power (kW)'].values * 1000
}).to_csv(os.path.join(output_dir, 'DHW_THESS.csv'), index=False)

########################################################################################################

# 3. Θέρμανση Χώρου
pd.DataFrame({
    'datetime': df.index,
    'Q_space_W': df['HVAC Heating Electric Power (kW)'].values * 1000
}).to_csv(os.path.join(output_dir, 'SPACE_HEAT_THESS.csv'), index=False)

########################################################################################################

# Ανάγνωση καιρού
weather_file='/content/GRC_Thessaloniki.166220_IWEC.epw'
weather_df, _ = read_epw(weather_file)
weather_df = weather_df.iloc[:len(df)]

########################################################################################################

# 4. Solar Data (GHI)
pd.DataFrame({
    'datetime': df.index,
    'GHI': weather_df['ghi'].values
}).to_csv(os.path.join(output_dir, 'SOLAR_DATA_THESS.csv'), index=False)

########################################################################################################

# 5. Temperatures (Ambient & Collector)
pd.DataFrame({
    'datetime': df.index,
    'Tamb_C': weather_df['temp_air'].values,
    'Tcoll_C': weather_df['temp_air'].values + 5
}).to_csv(os.path.join(output_dir, 'TEMPERATURES_THESS.csv'), index=False)

print("OK")

In [ ]:
import os
import numpy as np
import pandas as pd
import datetime as dt
from ochre import Dwelling
from pvlib.iotools import read_epw
from ochre.utils import default_input_path
########################################################################################################

dwelling = Dwelling(start_time=dt.datetime(2018, 1, 1, 0, 0), time_res=dt.timedelta(minutes=60), duration=dt.timedelta(days=365), hpxml_file='/content/house1_ath.xml', weather_file='/content/GRC_Athens.167160_IWEC.epw')

df, metrics, hourly = dwelling.simulate()

########################################################################################################

# SEED

output_dir = 'HOUSE1'
os.makedirs(output_dir, exist_ok=True)

seed = 22
rng = np.random.default_rng(seed)

def noise(column, sigma):
    base = df[column].to_numpy() * 1000
    noise = rng.normal(0, sigma, len(base))
    return np.clip(base * (1 + noise), 0, None)

electricity_w = noise('Total Electric Power (kW)', 0.03)
dhw_w = noise('Water Heating Electric Power (kW)', 0.03)
space_heat_w = noise('HVAC Heating Electric Power (kW)', 0.03)


########################################################################################################

# EXPORTS

# 1. Ηλεκτρισμός
pd.DataFrame({
    'datetime': df.index,
    'load_W': electricity_w
}).to_csv(os.path.join(output_dir, 'ELECTRICITY_ATH.csv'), index=False)

########################################################################################################

# 2. Ζεστό Νερό
pd.DataFrame({
    'datetime': df.index,
    'dhw_W': dhw_w
}).to_csv(os.path.join(output_dir, 'DHW_ATH.csv'), index=False)

########################################################################################################

# 3. Θέρμανση Χώρου
pd.DataFrame({
    'datetime': df.index,
    'Q_space_W': space_heat_w
}).to_csv(os.path.join(output_dir, 'SPACE_HEAT_ATH.csv'), index=False)

########################################################################################################

# Ανάγνωση καιρού
weather_file='/content/GRC_Athens.167160_IWEC.epw'
weather_df, _ = read_epw(weather_file)
weather_df = weather_df.iloc[:len(df)]

########################################################################################################

# 4. Solar Data (GHI)
pd.DataFrame({
    'datetime': df.index,
    'GHI': weather_df['ghi'].values
}).to_csv(os.path.join(output_dir, 'SOLAR_DATA_ATH.csv'), index=False)

########################################################################################################

# 5. Temperatures (Ambient & Collector)
pd.DataFrame({
    'datetime': df.index,
    'Tamb_C': weather_df['temp_air'].values,
    'Tcoll_C': weather_df['temp_air'].values + 5
}).to_csv(os.path.join(output_dir, 'TEMPERATURES_ATH.csv'), index=False)

print("OK")